# Evaluation: hybrid retrieval, cited answers and refusals

Reads `artifacts/eval_results.json` (written by `python -m docqa.run_eval`) and the hand-written `eval/qa_set.json`. Nothing here is re-tuned: it only reports what the runs measured.

In [1]:
import json, pandas as pd
from pathlib import Path
ROOT = Path('..')
res = json.loads((ROOT/'artifacts'/'eval_results.json').read_text(encoding='utf-8'))
qa = pd.DataFrame(json.loads((ROOT/'eval'/'qa_set.json').read_text(encoding='utf-8')))
chunks = pd.read_parquet(ROOT/'artifacts'/'chunks.parquet')
print(len(chunks), 'chunks;', chunks.n_chars.median(), 'median chars;', chunks.page.nunique(), 'distinct page numbers')
qa.groupby(['answerable','split']).size().unstack()

1379 chunks; 949.0 median chars; 95 distinct page numbers


split,dev,test
answerable,,
False,2,6
True,10,30


## The question set
Answerable questions per document and the gold-page spread:

In [2]:
a = qa[qa.answerable]
print(a.groupby('doc').size().to_string())
print('gold pages per question:', a.gold_pages.map(len).value_counts().sort_index().to_dict())

doc
ai-genai      6
ai-rmf        6
csf           6
identity      4
incident      4
privacy       4
ssdf          5
zero-trust    5
gold pages per question: {1: 32, 2: 8}


## Retrieval
`hit@k` is strict (annotated gold page); `suff@k` asks whether the top-k chunks together contain every required fact.

In [3]:
r = pd.DataFrame({m: {k: v for k, v in d.items() if isinstance(v, float)} for m, d in res['retrieval'].items()}).T
r[['hit@1','hit@4','hit@10','mrr@10','suff@1','suff@4','suff@10']].round(3)

,hit@1,hit@4,hit@10,mrr@10,suff@1,suff@4,suff@10
bm25,0.450,0.650,0.850,0.563,0.550,0.775,0.85
dense,0.375,0.725,0.875,0.543,0.525,0.800,0.90
hybrid,0.425,0.725,0.925,0.578,0.550,0.800,0.95
hybrid+rerank,0.475,0.775,0.900,0.631,0.650,0.825,0.90


Where the best retriever (hybrid + rerank) still misses the gold page in its top 5:

In [4]:
pd.DataFrame(res['retrieval_misses_top5'])[['id','question','gold']]

,id,question,gold
0,a04,Is use of the AI RMF mandatory?,"[ai-rmf, [7]]"
1,a06,What are AI actors?,"[ai-rmf, [7]]"
2,a09,How does the profile define the Data Privacy r...,"[ai-genai, [8]]"
3,a13,What are the six Functions of the CSF 2.0 Core?,"[csf, [8]]"
4,a19,What are the five Functions of the Privacy Fra...,"[privacy, [10]]"
5,a24,What does SSDF practice PS.1 aim to prevent?,"[ssdf, [18]]"
6,a37,How does SP 800-63-4 change IAL1?,"[identity, [15]]"
7,a38,What new authentication options does SP 800-63...,"[identity, [15]]"


## Answers on the held-out test split

In [5]:
g = pd.DataFrame({s: {k: v for k, v in d['test'].items() if isinstance(v, float)} for s, d in res['generation'].items()}).T
g.round(3)

,complete_rate,mean_coverage,false_abstain_rate,cited_rate,citation_precision,any_gold_cited_rate,invalid_citation_rate,model_cited_rate,correct_abstain_rate,mean_seconds
extractive,0.767,0.790,0.0,1.0,0.346,0.733,0.0,1.000,0.667,1.851
qwen2.5-0.5b,0.700,0.762,0.0,1.0,0.500,0.600,0.0,0.033,0.667,9.769
qwen2.5-1.5b,0.733,0.818,0.0,1.0,0.476,0.567,0.0,0.033,1.000,13.069


The refusal threshold was tuned on the dev split only:

In [6]:
print('abstain threshold =', round(res['abstain_threshold'], 3))
ans = pd.DataFrame(res['answers']['qwen2.5-1.5b']).merge(qa[['id','question','doc','kind']], on='id')
ans[~ans.answerable][['id','kind','abstained','reason','top_score','question']]

abstain threshold = -0.844


,id,kind,abstained,reason,top_score,question
40,u01,off,True,low_retrieval_score,-11.053,What is the recipe for a classic margherita pi...
41,u02,off,True,low_retrieval_score,-9.532,Who won the 2022 FIFA World Cup?
42,u03,near,True,low_retrieval_score,-10.772,What is the maximum fine for a GDPR violation?
43,u04,near,True,model_abstained,2.770,Which specific vendor products does NIST certi...
44,u05,near,True,low_retrieval_score,-6.952,What is the required minimum password length u...
45,u06,near,True,low_retrieval_score,-1.753,How much does a NIST Cybersecurity Framework c...
46,u07,near,True,low_retrieval_score,-5.818,What did the HIPAA Security Rule require for a...
47,u08,near,True,model_abstained,0.517,What GPU hardware is recommended for training ...


## What the 1.5B model got wrong (test split, answerable, incomplete)
Every one is a real mistake worth reading: fabricated list items, misread passages.

In [7]:
bad = ans[(ans.answerable) & (ans.split=='test') & (~ans.complete.astype(bool))]
for _, x in bad.iterrows():
    print(x['id'], '|', x['question'])
    print('   ->', x['text'][:240].replace(chr(10), ' '))
    print()

a04 | Is use of the AI RMF mandatory?
   -> No, use of the AI RMF alone will not lead to the establishment of appropriate accountability mechanisms, roles and responsibilities, culture, and incentive structures for risk management. [1] Effective risk management is realized through or

a11 | What does the Value Chain and Component Integration risk refer to?
   -> The Value Chain and Component Integration risk refers to the risks associated with the value chain and component integration of AI systems. [1][2] This includes risks related to data privacy, information security, harmful bias and homogeniz

a12 | What kind of harm does the Environmental Impacts risk describe?
   -> The Environmental Impacts risk describes harms to the environment and ecosystems. [1][2]

a19 | What are the five Functions of the Privacy Framework Core?
   -> The five Functions of the Privacy Framework Core are Identify-P, Govern-P, Control-P, Communicate-P, and Detect-P. [2]

a23 | What are the four groups of pr

## Latency
CPU only (8 cores), mean seconds per answer:

In [8]:
{s: round(d['all']['mean_seconds'], 1) for s, d in res['generation'].items()}

{'extractive': 1.9, 'qwen2.5-0.5b': 9.7, 'qwen2.5-1.5b': 13.1}